In [2]:
# from deltalake import write_deltalake
# import pandas as pd

# schema = {'transit_timestamp' : '1600-01-01', 
#           'transit_mode' : "-1", 
#           'station_complex_id' : "-1",
#        'station_complex': "-1", 
#        'borough' : '-1', 
#        'payment_method' : '-1', 
#        'fare_class_category' : '-1',
#        'ridership' : 0, 
#        'transfers' : 0, 
#        'latitude' : 0.0, 
#        'longitude' : 0.0, 
#        'georeference' : "-1" }

# pd.DataFrame(columns=schema)
# #write_deltalake()

,transit_timestamp,transit_mode,station_complex_id,station_complex,borough,payment_method,fare_class_category,ridership,transfers,latitude,longitude,georeference


In [1]:
import requests
import pandas as pd

# Function to fetch the maximum and minimum transit_timestamp
def fetch_min_max_dates():
    '''Query the date of the oldest and latest records from the dataset'''
    # Define the API endpoint with SoQL query to get min and max dates
    url = (
        "https://data.ny.gov/resource/wujg-7c2s.json"
        "?$select=min(transit_timestamp) as min_date, max(transit_timestamp) as max_date"
    )

    # Make the GET request
    response = requests.get(url)

    # Handle the response
    if response.status_code == 200:
        data = response.json()
        if data:
            min_date = data[0].get('min_date')
            max_date = data[0].get('max_date')
            return min_date, max_date
        else:
            raise Exception("No data returned from the API.")
    else:
        raise Exception(f"Failed to fetch data: {response.status_code}, {response.text}")

# Example usage
try:
    min_date, max_date = fetch_min_max_dates()
    print(f"Minimum Date: {min_date}")
    print(f"Maximum Date: {max_date}")
except Exception as e:
    print(e)


''''''''''''''''''''''''''''''''



## General purpose
def generate_year_month_list(min_date_str, max_date_str):
    # Convert string dates to datetime objects
    min_date = pd.to_datetime(min_date_str)
    max_date = pd.to_datetime(max_date_str)

    # Generate a date range with monthly frequency
    date_range = pd.date_range(
        start=min_date.replace(day=1),
        end=max_date.replace(day=1),
        freq='MS'  # Month Start frequency
    )

    # Create a list of (year, month) tuples
    year_month_list = [(date.year, date.month) for date in date_range]
    return year_month_list

# Generate the year-month list
year_month_list = generate_year_month_list(min_date, max_date)
print("Year-Month List:", year_month_list)



## General purpose
# Function to fetch data between a start date and end date
def fetch_data_by_date(start_date, end_date):
    # Define the API endpoint with date filtering
    url = (
        f"https://data.ny.gov/resource/wujg-7c2s.json"
        f"?$where=transit_timestamp >= '{start_date}' AND transit_timestamp <= '{end_date}'"
        f"&$limit=5000000"
    )

    # Make the GET request
    response = requests.get(url)

    # Handle the response
    if response.status_code == 200:
        return response.json()  # Return JSON response
    else:
        raise Exception(f"Failed to fetch data: {response.status_code}, {response.text}")

# Function usage
# start_date = f"{year_month_list[0][0]}-{year_month_list[0][1]}-01"
# end_date = f"{year_month_list[1][0]}-{year_month_list[1][1]}-01"

# try:
#     data = fetch_data_by_date(start_date, end_date)
#     #print(data)
# except Exception as e:
#     print(e)


Minimum Date: 2020-07-01T00:00:00.000
Maximum Date: 2024-12-19T23:00:00.000
Year-Month List: [(2020, 7), (2020, 8), (2020, 9), (2020, 10), (2020, 11), (2020, 12), (2021, 1), (2021, 2), (2021, 3), (2021, 4), (2021, 5), (2021, 6), (2021, 7), (2021, 8), (2021, 9), (2021, 10), (2021, 11), (2021, 12), (2022, 1), (2022, 2), (2022, 3), (2022, 4), (2022, 5), (2022, 6), (2022, 7), (2022, 8), (2022, 9), (2022, 10), (2022, 11), (2022, 12), (2023, 1), (2023, 2), (2023, 3), (2023, 4), (2023, 5), (2023, 6), (2023, 7), (2023, 8), (2023, 9), (2023, 10), (2023, 11), (2023, 12), (2024, 1), (2024, 2), (2024, 3), (2024, 4), (2024, 5), (2024, 6), (2024, 7), (2024, 8), (2024, 9), (2024, 10), (2024, 11), (2024, 12)]


In [2]:
import gc
gc.collect()


0

In [3]:
from pathlib import Path
# import dask.dataframe as dd
import polars as pl
import pandas as pd

folder_path = Path("/Users/bharatpremnath/nycdatafiles")

if any(folder_path.iterdir()):
    print("There are files in the folder.")
    df = pl.read_parquet(f'{folder_path}/*.parquet')
    latest_date = df['transit_timestamp'].max()
    min_date, max_date = fetch_min_max_dates()
    year_month_list = generate_year_month_list(latest_date, max_date)
    for i in range(len(year_month_list)-1):
        start_date = f"{year_month_list[i][0]}-{year_month_list[i][1]}-01"
        end_date = f"{year_month_list[i+1][0]}-{year_month_list[i+1][1]}-01"
        try:
            data = fetch_data_by_date(start_date, end_date)
            df = pl.DataFrame(data)
            df.write_parquet(f'{folder_path}/nyc_daily_ridership_{end_date}_.parquet')
            del df
        except Exception as e:
            print(e)

else:
    print("No files found in the folder.")
    min_date, max_date = fetch_min_max_dates()
    year_month_list = generate_year_month_list(min_date, max_date)
    for i in range(len(year_month_list)-1):
        start_date = f"{year_month_list[i][0]}-{year_month_list[i][1]}-01"
        end_date = f"{year_month_list[i+1][0]}-{year_month_list[i+1][1]}-01"
        try:
            data = fetch_data_by_date(start_date, end_date)
            df = pl.DataFrame(data)

            

            df.write_parquet(f'{folder_path}/nyc_daily_ridership_{end_date}_.parquet')
            del df
        except Exception as e:
            print(e)



There are files in the folder.


: 

In [3]:
data_validation = {
  "$schema": "http://json-schema.org/draft-07/schema#",
  "title": "MTA Subway Hourly Ridership API Data",
  "type": "object",
  "properties": {
    "transit_timestamp": {
      "type": "string",
      "format": "date-time",
      "description": "Timestamp payment took place, rounded down to the nearest hour."
    },
    "transit_mode": {
      "type": "string",
      "enum": ['subway', 'staten_island_railway', 'tram'],
      "description": "Mode of transit."
    },
    "station_complex_id": {
      "type": "string",
      "pattern": "^[a-zA-Z0-9]+$",
      "description": "Unique identifier for station complexes."
    },
    "station_complex": {
      "type": "string",
      "description": "Name of the subway complex, including routes in parentheses."
    },
    "borough": {
      "type": "string",
      "enum": ["Bronx", "Brooklyn", "Manhattan", "Queens",'Staten Island'],
      "description": "Borough serviced by the subway system."
    },
    "payment_method": {
      "type": "string",
      "enum": ["omny", "metrocard"],
      "description": "Payment method used to enter."
    },
    "fare_class_category": {
      "type": "string",
      "enum": [
        "Metrocard - Fair Fare",
        "Metrocard - Full Fare",
        "Metrocard - Other",
        "Metrocard - Seniors & Disability",
        "Metrocard - Students",
        "Metrocard - Unlimited 30-Day",
        "Metrocard - Unlimited 7-Day",
        "OMNY - Full Fare",
        "OMNY - Other",
        "OMNY - Seniors & Disabilities"
      ],
      "description": "Class of fare payment used for the trip."
    },
    "ridership": {
      "type": "string",
      "pattern": "^[0-9]+(\\.[0-9]+)?$",
      "description": "Total number of riders, represented as a stringified float."
    },
    "transfers": {
      "type": "string",
      "pattern": "^[0-9]+(\\.[0-9]+)?$",
      "description": "Number of free bus-to-subway or out-of-network transfers, represented as a stringified float."
    },
    "latitude": {
      "type": "string",
      "pattern": "^-?\\d+\\.\\d+$",
      "description": "Latitude of the specified subway complex, represented as a string."
    },
    "longitude": {
      "type": "string",
      "pattern": "^-?\\d+\\.\\d+$",
      "description": "Longitude of the specified subway complex, represented as a string."
    },
    "georeference": {
      "type": "object",
      "properties": {
        "type": {
          "type": "string",
          "enum": ["Point"],
          "description": "GeoJSON object type."
        },
        "coordinates": {
          "type": "array",
          "items": [
            {
              "type": "number",
              "minimum": -180,
              "maximum": 180,
              "description": "Longitude value."
            },
            {
              "type": "number",
              "minimum": -90,
              "maximum": 90,
              "description": "Latitude value."
            }
          ],
          "minItems": 2,
          "maxItems": 2,
          "description": "GeoJSON coordinates array [longitude, latitude]."
        }
      },
      "required": ["type", "coordinates"],
      "additionalProperties": False,
      "description": "GeoJSON object representing the geographical location."
    }
  },
  "required": [
    "transit_timestamp",
    "transit_mode",
    "station_complex_id",
    "station_complex",
    "borough",
    "payment_method",
    "fare_class_category",
    "ridership",
    "transfers",
    "latitude",
    "longitude",
    "georeference"
  ],
  "additionalProperties": False
}

In [1]:
from jsonschema import validate, ValidationError
from concurrent.futures import ProcessPoolExecutor

#Validate data against the schema
for i in data:
    try:
        validate(instance=i, schema=data_validation)
    except ValidationError as e:
        print("Validation error:", e.message)

# Validation function
# global validate_instance
# def validate_instance(record):
#         try:
#             validate(instance=record, schema=data_validation)
#             return {"record": record, "valid": True, "error": None}
#         except ValidationError as e:
#             return {"record": record, "valid": False, "error": e.message}
    

# if __name__ == "__main__":
#     with ProcessPoolExecutor() as executor:
#         results = list(executor.map(validate_instance, data))

#     for result in results:
#         if result["valid"]:
#             print(f"Valid record: {result['record']}")
#         else:
#             print(f"Validation error: {result['error']} in record {result['record']}")

NameError: name 'data' is not defined

### Schema Validation

In [28]:
import numpy as np
from joblib import Parallel, delayed

def schema_validation(df):
    schema = ['transit_timestamp', 'transit_mode', 'station_complex_id',
        'station_complex', 'borough', 'payment_method', 'fare_class_category',
        'ridership', 'transfers', 'latitude', 'longitude', 'georeference']

    df = pl.DataFrame(data)

    # Schema validation
    missing_columns = set(schema) - set(df.columns)
    extra_columns = set(df.columns) - set(schema)

    if missing_columns or extra_columns:
        yield('Change in schema detected')
        if missing_columns:
            yield(f'Missing columns: {missing_columns}')
        if extra_columns:
            yield(f'Extra columns: {extra_columns}')
    else:
        return('Schema is consistent')

Schema is consistent


In [29]:
df['transit_mode'].unique()

array(['subway', 'staten_island_railway', 'tram'], dtype=object)

In [6]:
# Data Completeness
df.isnull().sum()
print('Data is complete')

import hashlib

def hash_dict(d):
    """Hash a dictionary (including nested structures)."""
    return hashlib.md5(str(sorted(d.items())).encode()).hexdigest()

# Handling Duplicates
dedupe_array = Parallel(n_jobs=-1)(delayed(hash_dict)(x) for x in np.array(data))
orig_len = len(data)

# Convert results back to NumPy array
dedupe_len = len(np.unique(np.array(dedupe_array)))

if dedupe_len != orig_len:
    print('Duplicates found')
else:
    print('No Duplicates found')

Data is complete
No Duplicates found
